In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [3]:
# Time
start = dt.datetime(2019,4,5)
end = dt.datetime(2019,5,25)
print(start,end,end-start)

2019-04-05 00:00:00 2019-05-25 00:00:00 50 days, 0:00:00


In [4]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}},{"sign_up_details":1, "created_at":1,"login_details":1}):
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
df_users = pd.DataFrame(dic_flattened)
df_users = df_users[df_users["sign_up_details_app_platform"] == "UNITY_Android"]
users = df_users[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","createtime","device_id","last_request"]
users.head()

,user_id,createtime,device_id,last_request
0,5ca6adb3b65b15544e169963,2019-04-05 01:21:55.931,ba643254ecf37ea8a8453a12ab14e7a8,2019-04-05 01:22:16.059
1,5ca6c1bc8c899454486ac253,2019-04-05 02:47:24.829,3ac01ce3f63c2c857064702d99c80541,2019-04-05 03:09:53.215
2,5ca6d01fd468e03534a80f6d,2019-04-05 03:48:47.021,2534dd971bd5601f2985c94d1da74cc9,2019-04-05 03:51:56.000
4,5ca6e3fc800ebb353a469cd1,2019-04-05 05:13:32.630,bdfe19ca2338a8176670c5c79695b36f,2019-04-05 10:11:34.754
5,5ca6e9a3acb9c30617a1e75f,2019-04-05 05:37:39.265,a395477c3bd6830b7e0c68571da67ed8,2019-04-05 05:37:44.384


In [5]:
users.sort_values(['device_id','createtime'],inplace = True)

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


In [6]:
users = users.drop_duplicates('device_id')

In [7]:
users = users[users['last_request']-users['createtime']>'360:00:00']

In [8]:
team_cursor = cursor.superstars.teams
aw_teams = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1, "created_at":1}):
    aw_teams.append(documents)
dic_flattened = [flatten(d) for d in aw_teams]
df_teams = pd.DataFrame(dic_flattened)
teams = df_teams[df_teams["user"].isin(users["user_id"])]
teams = teams[["_id","user","created_at"]]
teams.columns = ["team_id", "user_id", "team_created_at"]
teams.head()

,team_id,user_id,team_created_at
54,5ca6ed54d468e03534a811f7,5ca6ed54d468e03534a811d2,2019-04-05 05:53:24.507
79,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.458
131,5ca6f187f7fc491098bc4dfa,5ca6f187f7fc491098bc4dd5,2019-04-05 06:11:19.322
144,5ca6f21e42b61e76cb9373d5,5ca6f21d42b61e76cb9373b0,2019-04-05 06:13:50.059
241,5ca6f74b1730d418f0aaca9c,5ca6f74b1730d418f0aaca77,2019-04-05 06:35:55.758


In [9]:
users_team = pd.merge(teams,users,on='user_id')

In [10]:
users_team.head()

,team_id,user_id,team_created_at,createtime,device_id,last_request
0,5ca6ed54d468e03534a811f7,5ca6ed54d468e03534a811d2,2019-04-05 05:53:24.507,2019-04-05 05:53:24.410,3c6ba7bf8141ea0126bcdcd4d5892737,2019-05-07 10:15:42.682
1,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.458,2019-04-05 05:58:59.360,0f331606133063e1b259def52ffbccb3,2019-04-26 06:40:56.932
2,5ca6f187f7fc491098bc4dfa,5ca6f187f7fc491098bc4dd5,2019-04-05 06:11:19.322,2019-04-05 06:11:19.232,2d5c5e5262bf1965fb6f7b7e193be1e3,2019-04-28 15:59:09.483
3,5ca6f21e42b61e76cb9373d5,5ca6f21d42b61e76cb9373b0,2019-04-05 06:13:50.059,2019-04-05 06:13:49.946,e4281053bae83284da29b783f8050d27,2019-05-21 01:33:23.883
4,5ca6f74b1730d418f0aaca9c,5ca6f74b1730d418f0aaca77,2019-04-05 06:35:55.758,2019-04-05 06:35:55.668,356a4edbdb488403e4d8ecd393bb683f,2019-05-22 11:34:29.190


In [11]:
con_cursor = cursor.superstars.matches
aw_matches = []
for documents in con_cursor.find({'created_at': {'$lt': end, '$gte': start},"status":3,"type":"CAMPAIGN"}):
    aw_matches.append(documents)
dic_flattened = [flatten(d) for d in aw_matches]
df_matches = pd.DataFrame(dic_flattened)
df_matches = df_matches.rename(columns={'home_team_id':'team_id'})
matches = df_matches[df_matches["team_id"].isin(teams["team_id"])]
matches = matches.loc[:,["team_id","winner_team_id","status","type",'start_time']]
#matches.head()

In [12]:
matches = matches[matches['winner_team_id']==matches['team_id']]

In [13]:
users_match = pd.merge(matches,users_team,on='team_id')

In [14]:
users_match.head()

,team_id,winner_team_id,status,type,start_time,user_id,team_created_at,createtime,device_id,last_request
0,5ca6ed54d468e03534a811f7,5ca6ed54d468e03534a811f7,3,CAMPAIGN,2019-04-05 05:53:46.528,5ca6ed54d468e03534a811d2,2019-04-05 05:53:24.507,2019-04-05 05:53:24.410,3c6ba7bf8141ea0126bcdcd4d5892737,2019-05-07 10:15:42.682
1,5ca6ed54d468e03534a811f7,5ca6ed54d468e03534a811f7,3,CAMPAIGN,2019-04-05 05:57:37.494,5ca6ed54d468e03534a811d2,2019-04-05 05:53:24.507,2019-04-05 05:53:24.410,3c6ba7bf8141ea0126bcdcd4d5892737,2019-05-07 10:15:42.682
2,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a46a000,3,CAMPAIGN,2019-04-05 05:59:22.050,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.458,2019-04-05 05:58:59.360,0f331606133063e1b259def52ffbccb3,2019-04-26 06:40:56.932
3,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a46a000,3,CAMPAIGN,2019-04-05 06:11:30.194,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.458,2019-04-05 05:58:59.360,0f331606133063e1b259def52ffbccb3,2019-04-26 06:40:56.932
4,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a46a000,3,CAMPAIGN,2019-04-05 06:15:48.624,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.458,2019-04-05 05:58:59.360,0f331606133063e1b259def52ffbccb3,2019-04-26 06:40:56.932


In [15]:
users_match.drop(['status','type','team_created_at','device_id','last_request'],axis=1,inplace=True)

In [16]:
users_match = users_match[users_match['start_time']-users_match['createtime']<'360:00:00']

In [17]:
users_match.head()

,team_id,winner_team_id,start_time,user_id,createtime
0,5ca6ed54d468e03534a811f7,5ca6ed54d468e03534a811f7,2019-04-05 05:53:46.528,5ca6ed54d468e03534a811d2,2019-04-05 05:53:24.410
1,5ca6ed54d468e03534a811f7,5ca6ed54d468e03534a811f7,2019-04-05 05:57:37.494,5ca6ed54d468e03534a811d2,2019-04-05 05:53:24.410
2,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a46a000,2019-04-05 05:59:22.050,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.360
3,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a46a000,2019-04-05 06:11:30.194,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.360
4,5ca6eea3800ebb353a46a000,5ca6eea3800ebb353a46a000,2019-04-05 06:15:48.624,5ca6eea3800ebb353a469fdb,2019-04-05 05:58:59.360


In [18]:
matches_played = users_match.groupby('user_id')['start_time'].count()
matches_played = matches_played.to_frame().reset_index()

In [19]:
matches_played.head()

,user_id,start_time
0,5ca6ed54d468e03534a811d2,2
1,5ca6eea3800ebb353a469fdb,46
2,5ca6f187f7fc491098bc4dd5,17
3,5ca6f21d42b61e76cb9373b0,15
4,5ca6f74b1730d418f0aaca77,3


In [20]:
matches_played.columns = ['user_id','matches']

In [21]:
matches_played.describe()

,matches
count,3852.000000
mean,14.983385
std,17.228714
min,1.000000
25%,2.000000
50%,8.000000
75%,22.000000
max,118.000000


In [ ]:
len(matches_played)